In [1]:
# Cài tất cả thư viện cần thiết
!pip install transformers seqeval torch -q

In [2]:
!pip install datasets==2.14.7

In [4]:
!pip install conllu -q

In [5]:
from datasets import load_dataset

# Option 1: Universal Dependencies (khuyến nghị, dễ dùng hơn)
dataset = load_dataset("universal_dependencies", "en_ewt")

# Option 2: CoNLL-2003 (phổ biến, dùng cho cả NER)
# dataset = load_dataset("conll2003")

# Xem cấu trúc dataset
print(dataset)
print(dataset["train"].features)

Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/12543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2002 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2077 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 12543
    })
    validation: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 2002
    })
    test: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 2077
    })
})
{'idx': Value(dtype='string', id=None), 'text': Value(dtype='string', id=None), 'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'lemmas': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'upos': Sequence(feature=ClassLabel(names=['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX'], id=None), length=-1, id=None), 'xpos': Sequence(feature=Value(dt

In [6]:
# Xem 1 mẫu dữ liệu
sample = dataset["train"][0]
print("Tokens:", sample["tokens"])
print("POS tags:", sample["upos"])  # hoặc "pos_tags" tuỳ dataset

Tokens: ['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
POS tags: [10, 1, 10, 1, 6, 0, 16, 10, 10, 10, 1, 10, 1, 8, 0, 2, 8, 0, 2, 8, 0, 2, 10, 1, 2, 8, 6, 0, 1]


In [7]:
# Lấy danh sách nhãn
label_list = dataset["train"].features["upos"].feature.names
print("Labels:", label_list)
print("Num labels:", len(label_list))

Labels: ['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX']
Num labels: 18


In [8]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
def tokenize_and_align_labels(examples):
    # ✅ QUAN TRỌNG: is_split_into_words=True vì input đã là list tokens
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                # [CLS] và [SEP] → gán -100, loss sẽ bỏ qua
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Subword ĐẦU TIÊN của từ → gán nhãn thật
                label_ids.append(label[word_idx])
            else:
                # Subword tiếp theo (##...) → -100 để bỏ qua
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [10]:
# Áp dụng cho toàn bộ dataset (batched=True = nhanh hơn nhiều)
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True
)
print(tokenized_datasets)

Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2002 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 12543
    })
    validation: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2002
    })
    test: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2077
    })
})


In [11]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import numpy as np
from seqeval.metrics import classification_report

In [12]:
# 1. Tạo model với số nhãn đúng
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list)
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
# 2. DataCollator — tự động pad sequences trong batch
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [14]:
# 3. Compute metrics dùng seqeval
def compute_metrics(p):
    predictions, labels = p
    # Lấy class có xác suất cao nhất
    predictions = np.argmax(predictions, axis=2)

    # ✅ Lọc -100 (special tokens + non-first subwords)
    true_predictions = [
        [label_list[pred] for pred, lbl in zip(prediction, label)
         if lbl != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[lbl] for pred, lbl in zip(prediction, label)
         if lbl != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = classification_report(true_labels, true_predictions, output_dict=True)
    return {
        "precision": results["weighted avg"]["precision"],
        "recall":    results["weighted avg"]["recall"],
        "f1":        results["weighted avg"]["f1-score"],
    }

In [15]:
# 4. Training Arguments
training_args = TrainingArguments(
    output_dir="./pos-bert-model",
    num_train_epochs=3,
    per_device_train_batch_size=16, # Số sample xử lý mỗi lần forward/backward trên 1 GPU/CPU trong lúc train
    per_device_eval_batch_size=16, # Tương tự nhưng dùng khi đánh giá (eval/validation)
    warmup_steps=500, # Trong 500 bước đầu tiên, learning rate tăng dần từ 0 → giá trị target (thay vì bắt đầu ngay ở LR cao). Giúp model ổn định ở giai đoạn đầu training.
    weight_decay=0.01, # L2 regularization — phạt các trọng số quá lớn để chống overfitting. Giá trị 0.01 là mức nhẹ, phổ biến với BERT
    eval_strategy="epoch",    # ✅ Mới (transformers ≥ 4.38) # Chạy evaluation sau mỗi epoch kết thúc
    save_strategy="epoch", # Lưu checkpoint sau mỗi epoch
    load_best_model_at_end=True, # Sau khi train xong, tự động load lại checkpoint tốt nhất
    logging_steps=50, # Ghi log (loss, LR, ...) vào console/TensorBoard sau mỗi 50 bước. Giảm xuống 10 nếu muốn theo dõi chi tiết hơn
    report_to="none",         # Tắt tích hợp với Weights & Biases (wandb), MLflow, TensorBoard, v.v. Nếu không tắt, transformers có thể hỏi hoặc tự gửi log lên cloud
)

```
LR
▲
│     /‾‾‾‾‾‾‾‾
│    /  warmup → decay
│   /
└──────────────→ steps
   500
```

In [17]:
# 5. Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args, # Truyền vào toàn bộ cấu hình từ TrainingArguments ở bước trước — epochs, batch size, LR warmup, save strategy, v.v.
    train_dataset=tokenized_datasets["train"], # Tập dữ liệu dùng để train
    eval_dataset=tokenized_datasets["validation"], # Tập dữ liệu dùng để đánh giá sau mỗi epoch
    data_collator=data_collator, # Hàm gom các sample rời rạc thành một batch. Nhiệm vụ chính là padding
    processing_class=tokenizer, # Truyền tokenizer vào để Trainer biết cách xử lý input khi cần
    compute_metrics=compute_metrics, # Hàm tự định nghĩa để tính các chỉ số đánh giá sau mỗi eval. Trainer truyền vào (predictions, labels) và hàm này trả về dict kết quả
)

```
tokenized_datasets
├── "train"       → model học từ đây
├── "validation"  → model được đánh giá ở đây  ← eval_dataset
└── "test"        → dùng sau cùng, không truyền vào Trainer
```


```
Trainer
├── model          → AI cần train
├── args           → "luật chơi" (epochs, LR, save...)
├── train_dataset  → dữ liệu để học
├── eval_dataset   → dữ liệu để kiểm tra
├── data_collator  → đóng gói batch + padding
├── processing_class → tokenizer đi kèm model
└── compute_metrics → thước đo chất lượng
```

In [18]:
# 6. Train!
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.108300,0.137556,0.956835,0.959560,0.958041
2,0.058900,0.118305,0.963583,0.965482,0.964511
3,0.027300,0.128193,0.965021,0.966963,0.965958


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

TrainOutput(global_step=2352, training_loss=0.2173623531125486, metrics={'train_runtime': 496.0526, 'train_samples_per_second': 75.857, 'train_steps_per_second': 4.741, 'total_flos': 1024440131249964.0, 'train_loss': 0.2173623531125486, 'epoch': 3.0})